In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Scraping") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

df = spark.read.format("mongo").load()

df.printSchema()
df.show(5, truncate=False)


root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------------+-------------------+-------+-------------------------------------+-----+
|_id                       |fecha_captura      |grupo  |identificador                        |valor|
+--------------------------+-------------------+-------+-------------------------------------+-----+
|{69cd34a2e54a09aeb97c7959}|2026-04-01 15:07:09|AutoTec|A Light in the Attic                 |51.77|
|{69cd34a2e54a09aeb97c795a}|2026-04-01 15:07:09|AutoTec|Tipping the Velvet                   |53.74|
|{69cd34a2e54a09aeb97c795b}|2026-04-01 15:07:09|AutoTec|Soumission                           |50.1 |
|{69cd34a2e54a09aeb97c795c}|2026-04-01 15:07:09|AutoTec|Sharp Objects                        |47.82|
|{69cd34a2e54a09aeb97c795d}|2026-04-01 15:07:09|

In [2]:
from pyspark.sql.functions import col

# --- Filtro de Calidad (Limpieza) ---
# Solo queremos registros que tengan un 'identificador' real y un 'valor' mayor a 0
df_filtrado = df.filter((col("identificador").isNotNull()) & (col("valor") > 0))

# Mostrar los resultados
print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 60
Registros válidos: 60


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim

# --- CONFIGURACIÓN DE SPARK PARA CARGAR DATOS DESDE MONGODB ---
spark = SparkSession.builder \
    .appName("Analisis_Scraping") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

# --- LEER LOS DATOS DESDE MONGODB ---
df = spark.read.format("mongo").load()

# --- LIMPIEZA DE LOS DATOS ---

# 1. Eliminar filas con valores nulos en 'identificador' o 'valor'
df_limpio = df.dropna(subset=["identificador", "valor"])

# 2. Filtrar registros donde 'valor' sea mayor a 0
df_limpio = df_limpio.filter(col("valor") > 0)

# 3. Eliminar duplicados basados en 'identificador'
df_limpio = df_limpio.dropDuplicates(["identificador"])

# 4. Limpiar espacios en la columna 'identificador'
df_limpio = df_limpio.withColumn("identificador", trim(col("identificador")))

# 5. Convertir la columna 'valor' a tipo float
df_limpio = df_limpio.withColumn("valor", col("valor").cast("float"))

# --- MUESTRA DE DATOS LIMPIOS ---
df_limpio.show()

# --- RESULTADO DE LA LIMPIEZA ---
print(f"Registros originales: {df.count()}")
print(f"Registros válidos después de la limpieza: {df_limpio.count()}")

# --- OPCIÓN DE GUARDADO EN FORMATO PARQUET (opcional) ---
# df_limpio.write.format("parquet").save("/path/to/save/resultado_limpieza")

+--------------------+-------------------+-------+--------------------+-----+
|                 _id|      fecha_captura|  grupo|       identificador|valor|
+--------------------+-------------------+-------+--------------------+-----+
|{69cd34a2e54a09ae...|2026-04-01 15:07:12|AutoTec|#HigherSelfie: Wa...|23.11|
|{69cd34a2e54a09ae...|2026-04-01 15:07:09|AutoTec|A Light in the Attic|51.77|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|Aladdin and His W...|53.13|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|America's Cradle ...| 22.5|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec| Behind Closed Doors|52.22|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|Birdsong: A Story...|54.64|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|          Black Dust|34.53|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|Chase Me (Paris N...|25.27|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|Foolproof Preserv...|30.52|
|{69cd34a2e54a09ae...|2026-04-01 15:07:11|AutoTec|     How Music

In [4]:
df.select("identificador", "valor").show()

+--------------------+-----+
|       identificador|valor|
+--------------------+-----+
|A Light in the Attic|51.77|
|  Tipping the Velvet|53.74|
|          Soumission| 50.1|
|       Sharp Objects|47.82|
|Sapiens: A Brief ...|54.23|
|     The Requiem Red|22.65|
|The Dirty Little ...|33.34|
|The Coming Woman:...|17.93|
|The Boys in the B...| 22.6|
|     The Black Maria|52.15|
|Starving Hearts (...|13.99|
|Shakespeare's Son...|20.66|
|         Set Me Free|17.46|
|Scott Pilgrim's P...|52.29|
|Rip it Up and Sta...|35.02|
|Our Band Could Be...|57.25|
|                Olio|23.88|
|Mesaerion: The Be...|37.59|
|Libertarianism fo...|51.33|
|It's Only the Him...|45.17|
+--------------------+-----+
only showing top 20 rows



In [5]:
df.filter(df["valor"] > 100).show()

+---+-------------+-----+-------------+-----+
|_id|fecha_captura|grupo|identificador|valor|
+---+-------------+-----+-------------+-----+
+---+-------------+-----+-------------+-----+



In [6]:
df.groupBy("grupo").count().show()

+-------+-----+
|  grupo|count|
+-------+-----+
|AutoTec|   60|
+-------+-----+



In [8]:
reporte_categorias = df.groupBy("grupo").agg(
    F.count("identificador").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR GRUPO:")
reporte_categorias.show()

ANALISIS DE MERCADO POR GRUPO:
+-------+---------------+-----------------+-------------+-------------+
|  grupo|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+-------+---------------+-----------------+-------------+-------------+
|AutoTec|             60|35.00266666666666|        12.84|        57.31|
+-------+---------------+-----------------+-------------+-------------+



In [9]:
from pyspark.sql import functions as F

# 1. Ordenar por valor de forma descendente
# 2. Tomar el primer registro
ganador = df.orderBy(F.desc("valor")).limit(1)

# 3. Seleccionar las columnas relevantes y mostrar los resultados
ganador.select("identificador", "valor", "grupo", "fecha_captura").show(truncate=False)

+------------------------------+-----+-------+-------------------+
|identificador                 |valor|grupo  |fecha_captura      |
+------------------------------+-----+-------+-------------------+
|Slow States of Collapse: Poems|57.31|AutoTec|2026-04-01 15:07:12|
+------------------------------+-----+-------+-------------------+



In [14]:
from pyspark.sql import functions as F

# Definir el nombre del grupo (asegúrate de asignar el nombre adecuado)
NOMBRE_GRUPO = "AutoTec"  # Cambia esto por el nombre de tu grupo

# CONSULTA DE SALIDA: Resumen de integridad por Grupo
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("grupo") \
    .agg(
        F.count("identificador").alias("total_libros"),  # Contar el total de libros por grupo
        F.round(F.avg("valor"), 2).alias("precio_medio"),  # Calcular el precio promedio por grupo
        F.max("fecha_captura").alias("ultima_sincronizacion")  # Obtener la última sincronización
    )

# Mostrar el Ticket de Salida con el resumen
print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: AutoTec ---
+-------+------------+------------+---------------------+
|  grupo|total_libros|precio_medio|ultima_sincronizacion|
+-------+------------+------------+---------------------+
|AutoTec|          60|        35.0|  2026-04-01 15:07:13|
+-------+------------+------------+---------------------+

